In [ ]:
%cd /content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/ESRGAN

/content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/ESRGAN


In [ ]:
!pip install -r requirements.txt
!python setup.py develop

In [ ]:
!pip install rasterio

In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

In [6]:
from basicsr.utils.options import yaml_load
opt = yaml_load('/content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/ESRGAN/options/train/ESRGAN/train_ESRGAN_x4_s2naip.yml')

In [7]:
opt['is_train'] = True
opt['dist'] = False
opt['name'] = 'ESRGAN_x4_s2naip'
opt['scale'] = 4

In [8]:
from basicsr.models import build_model
model_fix = build_model(opt)
opt['network_g']['num_in_ch']=4
# opt['network_d']['input_size'] = 512
model = build_model(opt)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 224MB/s]


In [9]:
model.net_d = model.net_d.to('cuda')
model.net_g = model.net_g.to('cuda')
model_fix.net_g = model_fix.net_g.to('cuda')

In [ ]:
def save_network(save_path, net, param_key='params'):
    """Save networks.

    Args:
            net (nn.Module | list[nn.Module]): Network(s) to be saved.
            param_key (str | list[str]): The parameter key(s) to save network.
                Default: 'params'.
    """
    net = net if isinstance(net, list) else [net]
    param_key = param_key if isinstance(param_key, list) else [param_key]
    assert len(net) == len(param_key), 'The lengths of net and param_key should be the same.'

    save_dict = {}
    for net_, param_key_ in zip(net, param_key):
        # net_ = self.get_bare_model(net_)
        state_dict = net_.state_dict()
        for key, param in state_dict.items():
            if key.startswith('module.'):  # remove unnecessary 'module.'
                key = key[7:]
            state_dict[key] = param.cpu()
        save_dict[param_key_] = state_dict

    torch.save(save_dict, save_path)

from copy import deepcopy

def _print_different_keys_loading(crt_net, load_net, strict=True):
        """Print keys with different name or different size when loading models.

        1. Print keys with different names.
        2. If strict=False, print the same key but with different tensor size.
            It also ignore these keys with different sizes (not load).

        Args:
            crt_net (torch model): Current network.
            load_net (dict): Loaded network.
            strict (bool): Whether strictly loaded. Default: True.
        """
        crt_net = crt_net.state_dict()
        crt_net_keys = set(crt_net.keys())
        load_net_keys = set(load_net.keys())

        if crt_net_keys != load_net_keys:
            print('Current net - loaded net:')
            for v in sorted(list(crt_net_keys - load_net_keys)):
                print(f'  {v}')
            print('Loaded net - current net:')
            for v in sorted(list(load_net_keys - crt_net_keys)):
                print(f'  {v}')

        # check the size for the same keys
        if not strict:
            common_keys = crt_net_keys & load_net_keys
            for k in common_keys:
                if crt_net[k].size() != load_net[k].size():
                    print(f'Size different, ignore [{k}]: crt_net: '
                                   f'{crt_net[k].shape}; load_net: {load_net[k].shape}')
                    load_net[k + '.ignore'] = load_net.pop(k)

def load_network(net, load_path, strict=False, param_key='params'):
        """Load network.

        Args:
            load_path (str): The path of networks to be loaded.
            net (nn.Module): Network.
            strict (bool): Whether strictly loaded.
            param_key (str): The parameter key of loaded network. If set to
                None, use the root 'path'.
                Default: 'params'.
        """
        load_net = torch.load(load_path, map_location=lambda storage, loc: storage)
        if param_key is not None:
            if param_key not in load_net and 'params' in load_net:
                param_key = 'params'
            load_net = load_net[param_key]
        # remove unnecessary 'module.'
        for k, v in deepcopy(load_net).items():
            if k.startswith('module.'):
                load_net[k[7:]] = v
                load_net.pop(k)
        _print_different_keys_loading(net, load_net, strict)
        net.load_state_dict(load_net, strict=strict)

load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/SR_pretrained_models/ESRGAN_g_x4_epoch_weights_finetune.pth')
load_network(model.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_g_x4_epoch_weights_finetune.pth')
load_network(model.net_d, '/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_d_x4_epoch_weights_finetune.pth')

In [11]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*12], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*4], dtype=tf.int64),
    }

    @tf.function
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [12, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [4, 1000, 1000])

        return lres_img, hres_img

    @tf.function
    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

In [12]:
def random_crop_lr_hr(img_lr, img_hr, lr_crop_size):
    """
    Random aligned crop for super-resolution pairs.

    img_lr: Tensor [B, C, 64, 64]
    img_hr: Tensor [B, C, 1024, 1024]
    lr_crop_size: int (e.g., 32)

    Returns:
        lr_crop: [B, C, lr_crop_size, lr_crop_size]
        hr_crop: [B, C, lr_crop_size*16, lr_crop_size*16]
    """
    scale = img_hr.shape[-1] // img_lr.shape[-1]  # 16

    _, _, H_lr, W_lr = img_lr.shape
    assert H_lr >= lr_crop_size and W_lr >= lr_crop_size

    top_lr = torch.randint(0, H_lr - lr_crop_size + 1, (1,)).item()
    left_lr = torch.randint(0, W_lr - lr_crop_size + 1, (1,)).item()

    top_hr = top_lr * scale
    left_hr = left_lr * scale

    lr_crop = img_lr[:, :, top_lr:top_lr+lr_crop_size,
                             left_lr:left_lr+lr_crop_size]

    hr_crop = img_hr[:, :, top_hr:top_hr+lr_crop_size*scale,
                             left_hr:left_hr+lr_crop_size*scale]

    return lr_crop, hr_crop

In [ ]:
import torch.nn.functional as F
import tensorflow as tf
import torch

filenames = ['/content/S2_NAIP_SR_cloudless.tfrecords']


ds = input_pipeline(filenames, batch_size=4, is_shuffle=True, is_train=True, is_repeat=False)
for epoch in range(30):
    print(f'Epoch {epoch}')
    scaler = torch.cuda.amp.GradScaler()

    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/3500.0
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')/255.0

        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
        lr, hr = random_crop_lr_hr(lr, hr, lr_crop_size=16)

        with torch.no_grad():
            output = model_fix.net_g(lr)
            output = torch.clamp(output, 0, 1)


        train_data = {'lq': output, 'gt': hr}
        model.update_learning_rate(step, warmup_iter=-1)
        model.feed_data(train_data)
        model.optimize_parameters(step)

        if step == 0:
            print(lr.shape)
            print(hr.shape)
            print(output.shape)
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            lr = lr.detach().cpu().numpy()
            output = output.detach().cpu().numpy()
            lr_show = np.transpose(lr[0], (1, 2, 0))
            axes[0].imshow(lr_show[:, :, 3:0:-1])
            axes[0].set_title('Low-Resolution')
            axes[0].axis('off')

            output_show = np.transpose(output[0], (1, 2, 0))
            axes[1].imshow(output_show[:, :, :3])
            axes[1].set_title('High-Resolution')
            axes[1].axis('off')

        if step % 500 == 0:
            print(f'Step {step}, Loss: {model.log_dict}')
                # torch.save(model, '/content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/ATD_x16_step.pt')

    save_network('/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_g_x16_epoch_weights_finetune.pth', model.net_g, param_key='params')
    save_network('/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_d_x16_epoch_weights_finetune.pth', model.net_d, param_key='params')


# Downstream

In [18]:
def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_gmodel,
                    finetuned_dmodel,
                    output_tfrecords,
                    model, model_fix):

    num_test = num_sample-num_training
    load_network(model.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_g_x16_epoch_weights_finetune.pth')
    load_network(model.net_d, '/content/drive/MyDrive/GeoSR_new/S2NAIP/ESRGAN_d_x16_epoch_weights_finetune.pth')

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([12*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([4*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [12, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [4, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)

            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch


    #--------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')
        scaler = torch.cuda.amp.GradScaler()

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/3500.0
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')/255.0

            lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
            lr, hr = random_crop_lr_hr(lr, hr, lr_crop_size=16)

            with torch.no_grad():
                output = model_fix.net_g(lr)
                output = torch.clamp(output, 0, 1)

            train_data = {'lq': output, 'gt': hr}
            model.update_learning_rate(step, warmup_iter=-1)
            model.feed_data(train_data)
            model.optimize_parameters(step)

            if step % 50 == 0:
                print(f'Step {step}, Loss: {model.log_dict}')

        save_network(finetuned_gmodel, model.net_g, param_key='params')
        save_network(finetuned_dmodel, model.net_d, param_key='params')

    #--------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/ 3500.0
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)

        with torch.no_grad():
            output = model_fix.net_g(lr)
            output = torch.clamp(output, 0, 1)
            output = model.net_g(output)
        output = output.detach().cpu().numpy()
        output = np.clip(output, 0, 1)
        output = (output * 255).astype(np.uint8)
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        for i in range(label.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(lr_show[:, :, 3:0:-1])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(output_show[:, :, :3])
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1000
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/content/USBuildingFootprints.tfrecords'
finetuned_gmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_g_x16_USBuildingFootprints_Finetune.pth'
finetuned_dmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_d_x16_USBuildingFootprints_Finetune.pth'
output_tfrecords = '/content/drive/MyDrive/GeoSR_new/USBuildingFootprints_ESRGAN_x16.tfrecords'
load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/downstream_models/ESRGAN_g_x4_USBuildingFootprints_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_fix)

In [ ]:
lres_size = 51
hres_size = 853
hres_size_4x = 256
label_size = 512
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/content/ChesapeakeRSC.tfrecords'
finetuned_gmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_g_x16_ChesapeakeRSC_Finetune.pth'
finetuned_dmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_d_x16_ChesapeakeRSC_Finetune.pth'
output_tfrecords = '/content/drive/MyDrive/GeoSR_new/ChesapeakeRSC_ESRGAN_x16.tfrecords'
load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/downstream_models/ESRGAN_g_x4_ChesapeakeRSC_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1200
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/content/VermontLC.tfrecords'
finetuned_gmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_g_x16_VermontLC_Finetune.pth'
finetuned_dmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_d_x16_VermontLC_Finetune.pth'
output_tfrecords = '/content/drive/MyDrive/GeoSR_new/VermontLC_ESRGAN_x16.tfrecords'
load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/downstream_models/ESRGAN_d_x4_VermontLC_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1000
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/content/RoadDetections.tfrecords'
finetuned_gmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_g_x16_RoadDetections_Finetune.pth'
finetuned_dmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_d_x16_RoadDetections_Finetune.pth'
output_tfrecords = '/content/drive/MyDrive/GeoSR_new/RoadDetections_ESRGAN_x16.tfrecords'
load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/downstream_models/ESRGAN_g_x4_RoadDetections_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_fix)

In [ ]:
lres_size = 26
hres_size = 427
hres_size_4x = 256  # 4x sr image size
label_size = 256 #Vermontlc
num_sample = 2000
num_training = 1600
finetune_tfrecords = ['/content/CHM.tfrecords']
finetuned_gmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_g_x16_CHM_Finetune.pth'
finetuned_dmodel = '/content/drive/MyDrive/GeoSR_new/ESRGAN_d_x16_CHM_Finetune.pth'
output_tfrecords = '/content/drive/MyDrive/GeoSR_new/CHM_ESRGAN_x16.tfrecords'
load_network(model_fix.net_g, '/content/drive/MyDrive/GeoSR_new/S2NAIP/downstream_models/ESRGAN_g_x4_CHM_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_fix)